# Chapter 7 &mdash; Brzozowski's Minimization on `blimp`, End to End

**Concept 12 of the Chapter 7 decomposition:** *A Complete Illustration of Brzozowski's Minimization on `blimp`*

The four steps on a bloated DFA, cross-checked against `min_dfa` at every stage.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7/Concept-Brzozowski-On-Blimp/Concept-Brzozowski-On-Blimp.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


Everything from this chapter, applied to one machine.

`blimp` is a deliberately bloated DFA. The four steps &mdash; **rev, det, rev, det** &mdash;
shrink it to the minimal machine, and at each stage you can check what the
intermediate object *is*:

* after step 1: an **NFA** for the reversed language;
* after step 2: a **DFA** for the reversed language, with no equivalent states;
* after step 3: an **NFA** for the original language;
* after step 4: the **minimal DFA**.

Cross-check the final answer against `min_dfa` with `iso_dfa` &mdash; by Myhill&ndash;Nerode
they must agree.

## 2. Definitions

### The bloated machine

In [ ]:
blimp = md2mc('''DFA
I   : 0 -> A
I   : 1 -> B
A   : 0 -> C
A   : 1 -> D
B   : 0 -> D
B   : 1 -> C
C   : 0 -> E
C   : 1 -> F1
D   : 0 -> F1
D   : 1 -> E
E   : 0 | 1 -> E
F1  : 0 | 1 -> F1
''')
print("|Q| =", len(blimp["Q"]), " F =", sorted(blimp["F"]))

### The pipeline, with a report at each stage

In [ ]:
def brz_report(D):
    stages = []
    x = D
    for i, (op, fn) in enumerate([('rev', rev_dfa), ('det', nfa2dfa),
                                  ('rev', rev_dfa), ('det', nfa2dfa)], 1):
        x = fn(x)
        kind = 'NFA' if 'Q0' in x else 'DFA'
        stages.append((i, op, kind, len(x["Q"]), x))
    return stages

<!-- nav-strip -->

---

&larr;&nbsp;[Ch7&nbsp;11.&nbsp;Reversal of a DFA Yields an NFA](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7/Concept-Reversal-Yields-NFA/Concept-Reversal-Yields-NFA.ipynb) &nbsp;&middot;&nbsp; [**Chapter 7** index](https://github.com/ganeshutah/Jove/blob/master/Chapter7/README.md) &nbsp;&middot;&nbsp; [Ch8&nbsp;1.&nbsp;Regular Expressions: Syntax, Denotation, and Precedence](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8/Concept-RE-Syntax-And-Denotation/Concept-RE-Syntax-And-Denotation.ipynb)&nbsp;&rarr;

---

## 3. Tests

The four stages, with sizes and machine types.

In [ ]:
stages = brz_report(blimp)
print("%-6s %-6s %-6s %s" % ("step", "op", "kind", "|Q|"))
print("%-6s %-6s %-6s %d" % ("0", "-", "DFA", len(blimp["Q"])))
for i, op, kind, n, _ in stages:
    print("%-6d %-6s %-6s %d" % (i, op, kind, n))
assert stages[0][2] == 'NFA' and stages[1][2] == 'DFA'
assert stages[2][2] == 'NFA' and stages[3][2] == 'DFA'

Stage 2 is a DFA for the **reversed** language &mdash; check it.

In [ ]:
half = stages[1][4]
from itertools import product
strs = [''.join(p) for k in range(10) for p in product('01', repeat=k)]
assert all(accepts_dfa(half, s) == accepts_dfa(blimp, s[::-1]) for s in strs)
print("after two steps the machine accepts exactly the reversed language")

Stage 4 is the minimal DFA for the **original** language.

In [ ]:
final = stages[3][4]
assert all(accepts_dfa(final, s) == accepts_dfa(blimp, s) for s in strs)
print("after four steps the language is back to the original")
print("|Q| : blimp %d -> Brzozowski %d" % (len(blimp["Q"]), len(final["Q"])))

Cross-check against `min_dfa` &mdash; Myhill&ndash;Nerode says they must be isomorphic.

In [ ]:
m = min_dfa(blimp)
print("min_dfa      : %d states" % len(m["Q"]))
print("Brzozowski   : %d states" % len(final["Q"]))
print("iso_dfa      :", iso_dfa(final, m))
assert len(final["Q"]) == len(m["Q"])
assert iso_dfa(final, m)

And `min_dfa_brz` packages the whole pipeline.

In [ ]:
assert iso_dfa(min_dfa_brz(blimp), m)
print("min_dfa_brz agrees with both.  Three independent routes, one minimal machine.")

## 4. Animation

The end of the pipeline: `blimp`, minimized.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(min_dfa_brz(blimp), FuseEdges=True)

## 5. Exercises


1. Run the pipeline on the *reverse* of `blimp`. Do you get the same state count?
2. Which stage is the expensive one, and why?
3. Build your own bloated DFA and check both minimizers agree.

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter7/Concept-Brzozowski-On-Blimp')